In [1]:
# Ensure we are running from the project root so all relative paths work
import os
from gbheatmodel.path import PROJECT_ROOT

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from gbheatmodel import run_heat_demand_pipeline
from gbheatmodel.path import path_in_data

In [3]:
# 🔧 User inputs for heating demand pipeline

weather_year = 2010        # weather data / patterns year
future_year = 2050         # target future year
FES_year = "2022"          # FES dataset version
FES_scenario = "LW"        # "LW", "CT", "ST", "FS", ...

# Optional: a grid to inspect later
grid_id = "-3.5_56.0"

In [4]:
# Run the pipeline (fast if profiles already exist)
results = run_heat_demand_pipeline(
    weather_year=weather_year,
    future_year=future_year,
    FES_year=FES_year,
    scenario=FES_scenario,
    skip_if_profiles_exist=True,
)

# Directory containing saved profiles
profiles_dir = path_in_data(
    "outputs",
    "heating_profiles",
    f"{weather_year}_{FES_year}_{FES_scenario}_{future_year}",
)

# Extract results
LA_overlaps     = results["LA_overlaps"]
df_merged       = results["df_merged"]
total_dwellings = results["total_dwellings"]
profiles        = results["profiles"]

print("✅ Heating demand pipeline finished.")

# --- Handle cases depending on skip_if_profiles_exist ---
if LA_overlaps is None:
    # Pipeline was skipped → profiles already exist on disk
    print("⚡ Existing profiles detected — pipeline skipped.")
    print(f"📁 Profiles directory: {profiles_dir}")

else:
    # Full pipeline ran → we have full results
    print(f"- LA overlaps rows        : {len(LA_overlaps)}")
    print(f"- Merged FES rows         : {len(df_merged)}")
    print(f"- Total dwellings rows    : {len(total_dwellings)}")
    # profiles is a dict in full run
    if isinstance(profiles, dict):
        print(f"- Number of grid profiles : {len(profiles)}")
    else:
        print(f"- Profiles object type    : {type(profiles)}")


✅ Saved: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel\data\outputs\heating_profiles\2010_2022_LW_2050\half_hourly_heat_profile_-0.25_50.75_2010_2022_LW_2050.csv
✅ Saved: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel\data\outputs\heating_profiles\2010_2022_LW_2050\half_hourly_heat_profile_-0.25_51.0_2010_2022_LW_2050.csv
✅ Saved: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel\data\outputs\heating_profiles\2010_2022_LW_2050\half_hourly_heat_profile_-0.25_51.25_2010_2022_LW_2050.csv
✅ Saved: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel\data\outputs\heating_profiles\2010_2022_LW_2050\half_hourly_heat_profile_-0.25_51.5_2010_2022_LW_2050.csv
✅ Saved: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel\data\outputs\h

In [7]:
grid_total = total_dwellings["Total"].sum()
hp_total   = total_dwellings["TotalHPs"].sum()

print("Grid-level Total:", grid_total)
print("Grid-level TotalHPs:", hp_total)


Grid-level Total: 32428321.483796693
Grid-level TotalHPs: 20808351.36088253


In [8]:
total_dwellings_total = total_dwellings["Total"].sum()
print("Total dwellings across GB (from grids):", total_dwellings_total)


Total dwellings across GB (from grids): 32428321.483796693


In [6]:
# Sum tech units per grid from total_dwellings
tech_units_total = (
    total_dwellings
    .drop(columns=["Grid_ID"], errors="ignore")
    .select_dtypes(include="number")
    .to_numpy()
    .sum()
)
print("Tech units used in scaling:", tech_units_total)


Tech units used in scaling: 87178502.67571747


In [ ]:
from gbheatmodel import heat_tech_distribution

df_FES = heat_tech_distribution.FES_heat_supply_data.year_FES_heat_supply(
    future_year,
    FES_year,
    FES_scenario,
)

df = heat_tech_distribution.FES_shapefile_LAs_matching.match_FES_and_shapefile_LAs(
    df_FES,
    future_year,
    FES_year,
    FES_scenario,
)

baseline_total = df["Total"].sum()
baseline_hps   = df["TotalHPs"].sum()

print("Baseline Total from FES (LA-level):", baseline_total)
print("Baseline TotalHPs from FES:", baseline_hps)


In [ ]:
from gbheatmodel import FES_scaling_factor
FES_scaling = float(FES_scaling_factor.FES_scaling_factor_calc(future_year, FES_year,FES_scenario))
FES_scaling

In [ ]:
from gbheatmodel import heat_tech_distribution

distributed_df = heat_tech_distribution.distribute_all_heating_techs_to_grids(df_merged)
distributed_df

In [ ]:
# 🔍 Inspect and plot one grid's total half-hourly heat profile

if grid_id not in profiles:
    raise ValueError(f"Grid ID {grid_id} not found in profiles. "
                     f"Available examples: {list(profiles.keys())[:5]}")

df = profiles[grid_id]  # columns = technologies

# Sum across technologies to get total heat demand
series_total = df.sum(axis=1)

# Compute average daily profile (48 half-hours)
daily_matrix = series_total.values.reshape(-1, 48)
daily_avg = daily_matrix.mean(axis=0)

time_axis = [f"{int(i/2):02d}:{'30' if i % 2 else '00'}" for i in range(48)]

plt.figure(figsize=(10, 4))
plt.plot(time_axis, daily_avg)
plt.xticks(rotation=90)
plt.title(f"Average daily heat profile for grid {grid_id} "
          f"({weather_year} weather, {future_year}, {FES_year}/{FES_scenario})")
plt.ylabel("Heat demand (units)")
plt.tight_layout()
plt.show()
